# 🌾 Rice Price Forecasting — Demo Pipeline Walkthrough

This notebook demonstrates all 10 steps of the QGA–QPSO hybrid pipeline using the sample dataset.

**Runtime**: ~10–15 minutes on CPU (QGA + QPSO steps are compute-intensive on the full dataset).

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print('✅ Imports OK')

## Step 1 — Load Sample Data & Descriptive Statistics

In [ ]:
from rice_price_forecasting import compute_descriptive_stats

# Using the bundled sample CSV (500 rows, 5 states)
SAMPLE_FILE = '../data/sample/sample_data.csv'
df_raw = pd.read_csv(SAMPLE_FILE)
df_raw['DATE_STD'] = pd.to_datetime(df_raw['DATE_STD'], dayfirst=True)
print(df_raw.head())
print(f'\nShape: {df_raw.shape}')
print(f'States: {df_raw.STATE_KEY.unique()}')

## Step 2 — ACF & Periodogram Analysis

In [ ]:
from rice_price_forecasting import (
    build_series, make_stationary, plot_acf_figure, plot_periodogram
)

series_list, series_names = build_series(df_raw)
stationary_list, labels = zip(*[make_stationary(s) for s in series_list])
plot_acf_figure(list(stationary_list), series_names, labels[0])
plot_periodogram(list(stationary_list), series_names, labels[0])

## Step 3 — Feature Engineering (43 features)

In [ ]:
from rice_price_forecasting import engineer_features

df_fe = engineer_features(df_raw)
print(f'Features created: {df_fe.shape[1] - 3}')  # subtract STATE_KEY, DATE_STD, y
print(df_fe.columns.tolist())

## Step 4 — Train / Val / Test Split (state-stratified)

In [ ]:
from rice_price_forecasting import split_statewise, build_expanding_folds

df_fe['DATE_STD'] = pd.to_datetime(df_fe['DATE_STD'])
df_fe = df_fe.sort_values(['STATE_KEY', 'DATE_STD']).reset_index(drop=True)

trainval_idx, test_idx, split_summary = split_statewise(df_fe, test_frac=0.20, min_rows=30)
print(split_summary[['STATE_KEY', 'ROWS', 'STATUS']])
print(f'\nTrainVal: {len(trainval_idx)} | Test: {len(test_idx)}')

## Step 5 — QGA Feature Selection

In [ ]:
# NOTE: Full QGA runs 28 particles × 18 generations × XGBoost evaluations.
# Here we use reduced settings for the demo (fast ~1 min).
from rice_price_forecasting import (
    run_qga, internal_split_statewise, get_Xy
)
import rice_price_forecasting as rp

df_tv = df_fe.loc[trainval_idx].copy().sort_values(['STATE_KEY', 'DATE_STD']).reset_index(drop=True)
df_tr_int, df_va_int = internal_split_statewise(df_tv, val_frac=0.2, min_rows=30)

drop_cols = ['STATE_KEY', 'DATE_STD', 'y']
feature_cols_full = [c for c in df_tv.columns if c not in drop_cols]

# Inject data into module globals for QFS fitness function
rp.Xtr_full = df_tr_int[feature_cols_full].to_numpy(dtype='float32')
rp.ytr      = df_tr_int['y'].to_numpy(dtype='float32')
rp.Xva_full = df_va_int[feature_cols_full].to_numpy(dtype='float32')
rp.yva      = df_va_int['y'].to_numpy(dtype='float32')

# Reduced settings for demo
best_mask_qga, best_score = run_qga(pop=8, gens=5, elite_k=3)
QGA_features = [feature_cols_full[i] for i in np.where(best_mask_qga == 1)[0]]
print(f'\nQGA selected {len(QGA_features)} features:')
print(QGA_features)

## Step 6 — QPSO Hyperparameter Optimisation

In [ ]:
from rice_price_forecasting import run_qpso, get_Xy
import json

Xtr_qga, ytr = get_Xy(df_tr_int, QGA_features)
Xva_qga, yva = get_Xy(df_va_int, QGA_features)

# Reduced settings for demo
best_params = run_qpso(Xtr_qga, ytr, Xva_qga, yva, n_particles=5, iters=5)
print('\nBest XGBoost params:')
print(json.dumps(best_params, indent=2))

## Step 7 — Hybrid Model Training & Evaluation

In [ ]:
from rice_price_forecasting import train_hybrid

df_trainval = df_fe.loc[trainval_idx].copy()
df_test     = df_fe.loc[test_idx].copy()
n = len(df_trainval)
df_train_h = df_trainval.iloc[:int(n*0.70)].copy()
df_val_h   = df_trainval.iloc[int(n*0.70):int(n*0.85)].copy()

xgb_trend, res_model, test_metrics = train_hybrid(
    df_train_h, df_val_h, df_test, QGA_features
)
print('\n✅ Test metrics:')
for k, v in test_metrics.items():
    print(f'  {k:10s}: {v:.4f}')

## Step 8 — Multi-Horizon Forecast Evaluation (7 / 14 / 30 days)

In [ ]:
from rice_price_forecasting import evaluate_horizons

df_horizon = evaluate_horizons(df_test, QGA_features, xgb_trend, res_model)
print(df_horizon)